# 상품 추천 시스템 프로젝트

## 프로젝트 개요

**배경:** 이커머스 플랫폼에서 "이 상품 어때요?" 기능을 개발하기 위해, 유저의 구매 이력과 평점 데이터를 기반으로 개인화된 상품 추천 시스템을 구축합니다.

**목표:**
- 유저별 맞춤 상품 추천 리스트 생성
- 인기도 기반, 유저 기반 협업 필터링, 아이템 기반 협업 필터링 3가지 방식 구현 및 비교
- 비즈니스 적용 방안 제시

**데이터 설명:**
| 파일 | 설명 |
|------|------|
| `users.csv` | 유저 500명 (ID, 성별, 연령, 가입일, 선호카테고리) |
| `products.csv` | 상품 100개 (ID, 상품명, 카테고리, 가격, 평균평점) |
| `purchase_history.csv` | 구매이력 20,000건 (구매ID, 유저ID, 상품ID, 구매일, 평점, 구매금액) |


## 1. 데이터 로딩 & 탐색


#### 가이드: 환경 설정 및 라이브러리 임포트

다음 단계에 따라 분석 환경을 설정하세요.

**1단계: 필요 라이브러리 임포트**
- `pandas` → `pd` : 데이터프레임 처리
- `numpy` → `np` : 수치 연산
- `matplotlib.pyplot` → `plt` : 시각화
- `matplotlib` → `mpl` : 폰트 설정용
- `cosine_similarity` : `sklearn.metrics.pairwise`에서 임포트 (유사도 계산)

**2단계: 한글 폰트 설정**
- `platform.system()`으로 OS를 판별합니다
- Mac(`'Darwin'`): `AppleGothic` 폰트 사용
- Windows: `Malgun Gothic` 폰트 사용
- 마이너스 부호 깨짐 방지: `plt.rcParams['axes.unicode_minus'] = False`

**3단계: 데이터 경로 설정**
- `DATA_DIR` 변수에 `"./data/"` 경로를 지정합니다

사용할 함수/메서드: `plt.rc()`, `plt.rcParams()`, `rc()`, `system()`


In [ ]:
# 아래에 코드를 작성하세요


#### 가이드: 데이터 로딩

3개의 CSV 파일을 `pd.read_csv()`로 읽어옵니다.

**1단계: 파일 로딩**
| 파일명 | 변수명 | 컬럼 |
|--------|--------|------|
| `users.csv` | `users` | 유저ID, 성별, 연령, 가입일, 선호카테고리 |
| `products.csv` | `products` | 상품ID, 상품명, 카테고리, 가격, 평균평점 |
| `purchase_history.csv` | `purchases` | 구매ID, 유저ID, 상품ID, 구매일, 평점, 구매금액 |

- 경로는 `DATA_DIR + '파일명'` 형태로 사용합니다

**2단계: 데이터 크기 확인**
- 각 데이터프레임의 `.shape`를 출력하여 행/열 수를 확인합니다

사용할 함수/메서드: `pd.read_csv()`, `read_csv()`


In [ ]:
# 데이터 로딩

# 아래에 코드를 작성하세요


#### 가이드: 데이터 미리보기

각 데이터프레임의 `.head()`를 호출하여 상위 5행을 확인합니다.
- `users.head()` → 유저 정보 확인
- `products.head()` → 상품 정보 확인
- `purchases.head()` → 구매이력 확인

In [ ]:
# 아래에 코드를 작성하세요


In [ ]:
# 아래에 코드를 작성하세요


In [ ]:
# 아래에 코드를 작성하세요


#### 가이드: 데이터 타입 및 구조 확인

각 데이터프레임에 `.info()`를 호출하여 다음을 확인합니다:
- 컬럼별 데이터 타입 (object, int64, float64 등)
- Non-Null 개수 (결측치 존재 여부)
- 메모리 사용량

사용할 함수/메서드: `info()`


In [ ]:
# 아래에 코드를 작성하세요


## 2. 데이터 전처리


#### 가이드: 결측치 확인

**1단계: 각 데이터프레임의 전체 결측치 합계 확인**
- `.isnull().sum().sum()`으로 전체 결측치 건수를 확인합니다

**2단계: 구매이력 평점 결측 비율 확인**
- `purchases['평점']` 컬럼의 결측치 수와 비율(%)을 확인합니다
- `.isnull().sum()` → 결측 건수
- `.isnull().mean() * 100` → 결측 비율(%)

사용할 함수/메서드: `isnull()`, `mean()`, `sum()`


In [ ]:
# 결측치 확인

# 아래에 코드를 작성하세요


#### 가이드: 데이터 전처리 (날짜 변환 및 데이터 병합)

**1단계: 날짜형 변환**
- `users['가입일']`과 `purchases['구매일']`을 `pd.to_datetime()`으로 변환합니다

**2단계: 데이터 병합 (merge)**
- `purchases`에 `products` 정보를 `'상품ID'` 기준으로 left join합니다
- 그 결과에 `users` 정보를 `'유저ID'` 기준으로 left join합니다
- 최종 통합 데이터프레임 변수명: `df`

사용할 함수/메서드: `merge()`

**3단계: 결과 확인**
- `df.shape`로 통합 데이터 크기를 확인하고 `df.head()`로 미리보기합니다


In [ ]:
# 날짜 형변환

# 아래에 코드를 작성하세요


## 3. EDA


### 3-1. 카테고리별 판매 현황


#### 가이드: 카테고리별 판매 현황 분석 및 시각화

**1단계: 카테고리별 집계**
- `df.groupby('카테고리')`로 그룹핑 후 `.agg()`로 다음을 계산합니다:
  - `판매건수`: `('구매ID', 'count')`
  - `총매출`: `('구매금액', 'sum')`
  - `평균평점`: `('평점', 'mean')`
- `.round(2)`로 소수점 반올림, `sort_values('판매건수', ascending=False)`로 정렬
- 결과를 `cat_sales` 변수에 저장합니다

**2단계: 시각화 (1x2 subplot)**
- `fig, axes = plt.subplots(1, 2, figsize=(12, 5))`
- `axes[0]`: 카테고리별 판매 건수 막대그래프 (`'steelblue'`)
- `axes[1]`: 카테고리별 총매출(백만원 단위: `/ 1e6`) 막대그래프 (`'coral'`)

사용할 함수/메서드: `agg()`, `groupby()`, `round()`


In [ ]:
# 아래에 코드를 작성하세요


### 3-2. 인기 상품 Top 20


#### 가이드: 인기 상품 Top 20 시각화

**1단계: 상품별 집계**
- `df.groupby(['상품ID', '상품명'])`으로 그룹핑
- `판매건수`: `('구매ID', 'count')`, `평균평점`: `('평점', 'mean')`
- `.sort_values('판매건수', ascending=False).head(20)`으로 상위 20개 추출
- 결과를 `top20` 변수에 저장합니다

**2단계: 수평 막대그래프**
- `fig, ax = plt.subplots(figsize=(10, 7))`
- `ax.barh()`로 수평 막대 그래프를 그립니다
- y축 라벨: `상품ID + 상품명` 조합 (MultiIndex이므로 `top20.index`에서 추출)
- `ax.invert_yaxis()`로 상위 항목이 위에 오도록 합니다

사용할 함수/메서드: `barh()`, `invert_yaxis()`, `set_yticklabels()`, `set_yticks()`


In [ ]:
# 아래에 코드를 작성하세요


### 3-3. 유저별 구매 패턴


#### 가이드: 유저별 구매 패턴 분석

**1단계: 유저별 통계 집계**
- `df.groupby('유저ID')`로 그룹핑하여 다음을 계산합니다:
  - `구매수`: `('구매ID', 'count')`
  - `총지출`: `('구매금액', 'sum')`
  - `평균평점`: `('평점', 'mean')`
- 결과를 `user_stats` 변수에 저장합니다

**2단계: 시각화 (1x2 subplot)**
- `fig, axes = plt.subplots(1, 2, figsize=(12, 5))`
- `axes[0]`: 유저별 구매 건수 히스토그램 (bins=30)
  - 평균값을 빨간 점선(`axvline`)으로 표시합니다
- `axes[1]`: 유저 선호 카테고리 분포 막대그래프
  - `users['선호카테고리'].str.split(',').explode()`로 복수 선호를 분리하여 카운트합니다

사용할 함수/메서드: `explode()`, `split()`, `value_counts()`


In [ ]:
# 아래에 코드를 작성하세요


### 3-4. 평점 분포


#### 가이드: 평점 분포 시각화

**1단계: 평점별 건수 집계**
- `purchases['평점'].dropna().value_counts().sort_index()`로 평점별 건수를 집계합니다
- 결과를 `rating_counts` 변수에 저장합니다 (이후 대시보드에서도 재사용)

**2단계: 막대그래프**
- `fig, ax = plt.subplots(figsize=(8, 5))`
- x축: 평점(1~5, 정수), y축: 건수
- 각 막대 위에 건수를 텍스트로 표시합니다: `ax.text(i, v + 50, f"{v:,}", ha='center')`

**3단계: 통계 출력**
- 평균 평점: `purchases['평점'].mean()`
- 평점 결측 비율: `purchases['평점'].isnull().mean() * 100`

사용할 함수/메서드: `astype()`, `bar()`, `dropna()`, `sort_index()`, `value_counts()`


In [ ]:
# 아래에 코드를 작성하세요


## 4. 유저-상품 매트릭스 구성


#### 가이드: 유저-상품 평점 매트릭스 구성

**1단계: 평점 있는 데이터만 필터링**
- `purchases.dropna(subset=['평점'])`으로 평점 결측 행을 제거합니다
- 결과를 `ratings` 변수에 저장합니다

**2단계: 중복 구매 처리**
- 같은 유저가 같은 상품을 여러 번 구매한 경우 평균 평점을 사용합니다
- `ratings.groupby(['유저ID', '상품ID'])['평점'].mean().reset_index()` → `ratings_agg`

**3단계: 피벗 테이블로 매트릭스 생성**
- `user_item_matrix` 변수에 저장합니다

1. `pivot_table()`로 매트릭스 생성

**4단계: 희소성(Sparsity) 계산**
- 전체 셀 수: `행 수 x 열 수`
- 채워진 셀 수: `.notna().sum().sum()`
- 희소성 = `1 - (채워진 셀 / 전체 셀)`
- 희소성이 높을수록 데이터가 부족하여 추천이 어려움을 의미합니다


In [ ]:
# 평점이 있는 데이터만 사용하여 유저-상품 평점 매트릭스 생성

# 아래에 코드를 작성하세요


#### 가이드: 매트릭스 미리보기

`user_item_matrix.head()`를 호출하여 유저-상품 평점 매트릭스의 상위 5행을 확인합니다.
- 값이 있는 셀은 해당 유저가 해당 상품에 부여한 평점입니다
- `NaN`은 해당 유저가 해당 상품을 구매하지 않았거나 평점을 남기지 않은 것입니다

In [ ]:
# 아래에 코드를 작성하세요


## 5. 인기도 기반 추천


#### 가이드: 인기도 기반 추천 (전체 Top 10)

**1단계: 상품별 판매 통계 집계**
- `df.groupby(['상품ID', '상품명', '카테고리'])`로 그룹핑
- `판매건수`: `('구매ID', 'count')`, `평균평점`: `('평점', 'mean')`
- `.reset_index()`로 컬럼으로 복원
- 결과를 `popularity` 변수에 저장합니다

**2단계: 인기점수 계산**
- `인기점수 = 판매건수 x 평균평점` (단순 판매건수보다 평점 가중)
- 인기점수 기준 내림차순 정렬

사용할 함수/메서드: `sort_values()`

**3단계: Top 10 출력**
- `popularity[['상품ID', '상품명', '카테고리', '판매건수', '평균평점', '인기점수']].head(10)`


In [ ]:
# 전체 인기 상품 Top 10 (판매건수 x 평균평점 가중)

# 아래에 코드를 작성하세요


#### 가이드: 카테고리별 인기 상품 Top 5

- 5개 카테고리(`'전자'`, `'의류'`, `'식품'`, `'뷰티'`, `'생활'`)에 대해 반복합니다
- 각 카테고리에서 `popularity` 데이터프레임을 필터링하여 `.head(5)` 추출
- `['상품ID', '상품명', '판매건수', '평균평점']` 컬럼만 출력합니다

사용할 함수/메서드: `head()`, `to_string()`


In [ ]:
# 카테고리별 인기 상품 Top 5

# 아래에 코드를 작성하세요


## 6. 유저 기반 협업 필터링


#### 가이드: 유저 간 코사인 유사도 계산

**1단계: NaN을 0으로 대체**
- `user_item_matrix.fillna(0)` → `user_item_filled`
- 코사인 유사도 함수는 NaN을 처리하지 못하므로 0으로 채웁니다

**2단계: 코사인 유사도 계산**
- `cosine_similarity(user_item_filled)` → numpy 배열 반환
- 이를 DataFrame으로 변환하여 유저ID를 인덱스/컬럼으로 설정합니다

1. 결측치를 `fillna()`로 처리
2. `cosine_similarity()`로 유사도 계산

**3단계: 결과 확인**
- `user_sim_df.shape`로 매트릭스 크기를 확인합니다
- `.iloc[:5, :5].round(3)`으로 상위 5x5 미리보기합니다


In [ ]:
# 결측치를 0으로 채워서 코사인 유사도 계산

# 아래에 코드를 작성하세요


#### 가이드: 특정 유저와 유사한 유저 Top 5 확인

**1단계: 대상 유저 설정**
- `target_user = 'U0001'`

**2단계: 유사 유저 추출**
- `user_sim_df[target_user]`에서 자기 자신을 제외(`.drop(target_user)`)
- 유사도 내림차순 정렬 후 `.head(5)`

**3단계: 유사 유저 정보 출력**
- 각 유사 유저의 유사도 값과 선호카테고리를 함께 출력합니다
- `users[users['유저ID'] == uid]['선호카테고리'].values[0]`로 선호 조회

사용할 함수/메서드: `drop()`, `head()`, `sort_values()`


In [ ]:
# 특정 유저와 유사한 유저 Top 5

# 아래에 코드를 작성하세요


#### 가이드: 유저 기반 협업 필터링 추천 함수 구현

`recommend_user_based()` 함수를 다음 구조로 구현합니다:

**함수 시그니처:**


**내부 로직:**

1. **유사 유저 가져오기**: `user_sim_df[user_id]`에서 자기 자신 제외 후 유사도 상위 `n_similar`명 추출
2. **이미 구매한 상품 목록**: `user_item_matrix.loc[user_id].dropna().index.tolist()` → `user_rated`
3. **가중 평점 계산**: 유사 유저들의 평점에 유사도를 가중치로 곱합니다
   - `weighted_scores` 딕셔너리에 `{'score': 유사도*평점 합, 'sim_sum': 유사도 합}` 저장
   - 이미 구매한 상품(`user_rated`)은 제외합니다
4. **예측 평점 계산**: `score / sim_sum`으로 가중 평균 예측 평점 산출
5. **결과 반환**: 예측평점 상위 `n_recommend`개를 DataFrame으로 반환
   - `products_df`와 merge하여 `상품명`, `카테고리`, `가격` 정보를 추가합니다


In [ ]:
# 아래에 코드를 작성하세요


#### 가이드: 유저 기반 추천 결과 확인

- 샘플 유저 3명(`'U0001'`, `'U0050'`, `'U0200'`)에 대해 추천을 수행합니다
- 각 유저의 선호카테고리를 함께 출력하여 추천 결과와 비교합니다
- `recommend_user_based()` 함수를 호출하고 `['상품ID', '상품명', '카테고리', '예측평점']` 컬럼을 출력합니다



In [ ]:
# 추천 결과 예시 3명

# 아래에 코드를 작성하세요


## 7. 아이템 기반 협업 필터링


#### 가이드: 아이템 간 코사인 유사도 계산

**핵심 아이디어**: 유저-상품 매트릭스를 전치(`.T`)하면 상품x유저 매트릭스가 됩니다. 이를 이용해 상품 간 유사도를 계산합니다.

**1단계: 매트릭스 전치**
- `user_item_matrix.fillna(0).T` → `item_user_filled`

**2단계: 코사인 유사도 계산**
1. 결측치를 `fillna()`로 처리
2. `cosine_similarity()`로 유사도 계산

**3단계: 결과 확인**
- `item_sim_df.shape`와 `.iloc[:5, :5].round(3)` 미리보기


In [ ]:
# 상품 유사도 계산 (상품 x 유저 매트릭스 → 코사인 유사도)

# 아래에 코드를 작성하세요


#### 가이드: 특정 상품과 유사한 상품 Top 5 확인

**1단계: 대상 상품 설정 및 정보 조회**
- `target_item = 'P001'`
- `products`에서 상품명(`target_name`)과 카테고리(`target_cat`) 조회

**2단계: 유사 상품 추출**
- `item_sim_df[target_item]`에서 자기 자신 제외 후 유사도 상위 5개 추출

**3단계: 유사 상품 정보 출력**
- 각 상품의 상품명, 카테고리, 유사도를 함께 출력합니다

사용할 함수/메서드: `drop()`, `head()`, `sort_values()`


In [ ]:
# 특정 상품과 유사한 상품 Top 5

# 아래에 코드를 작성하세요


#### 가이드: 아이템 기반 협업 필터링 추천 함수 구현

`recommend_item_based()` 함수를 다음 구조로 구현합니다:

**함수 시그니처:**


**내부 로직:**
1. `item_sim_df[item_id]`에서 자기 자신 제외 후 유사도 상위 `n_recommend`개 추출
2. 상품ID와 유사도를 DataFrame으로 구성
3. `products_df`와 merge하여 `상품명`, `카테고리`, `가격`, `평균평점` 정보 추가
4. DataFrame 반환

사용할 함수/메서드: `DataFrame()`, `drop()`, `head()`, `merge()`, `pd.DataFrame()`, `sort_values()`


In [ ]:
# 아래에 코드를 작성하세요


#### 가이드: 아이템 기반 추천 결과 확인

- 샘플 상품 3개(`'P001'`, `'P025'`, `'P060'`)에 대해 추천을 수행합니다
- "이 상품을 보신 분은 이런 상품도 좋아합니다" 형태로 출력합니다
- `recommend_item_based()` 함수를 호출하고 `['상품ID', '상품명', '카테고리', '유사도']` 컬럼을 출력합니다



In [ ]:
# 추천 결과 예시 3개 상품

# 아래에 코드를 작성하세요


## 8. 추천 성능 평가


#### 가이드: 추천 성능 평가를 위한 Train/Test Split

**데이터 분할 전략**: 유저별 시간순 정렬 후 마지막 20%를 테스트셋으로 분리합니다.

**1단계: 데이터 정렬**
- `purchases.dropna(subset=['평점'])` → 평점 있는 데이터만 사용
- `sort_values(['유저ID', '구매일'])`로 유저별 시간순 정렬

**2단계: 유저별 80/20 분할**
- `groupby('유저ID')`로 유저별로 반복합니다
- 각 유저의 구매 이력에서 `int(n * 0.8)` 인덱스를 기준으로 분할
- 최소 1건은 train에 포함되도록 합니다

**3단계: 합치기**
- `pd.concat(train_list)` → `train`
- `pd.concat(test_list)` → `test`

사용할 함수/메서드: `append()`, `groupby()`


In [ ]:
# Train/Test Split: 유저별 마지막 20% 구매를 테스트셋으로

# 아래에 코드를 작성하세요


#### 가이드: Train 기반 모델 재구성

테스트 데이터 누출(data leakage)을 방지하기 위해 train 데이터만으로 모델을 다시 구성합니다.

**1단계: Train 기반 유저-상품 매트릭스**
- `train`에서 유저-상품별 평균 평점 → `train_agg`
- `pivot_table(index='유저ID', columns='상품ID', values='평점')` → `train_matrix`

**2단계: Train 기반 유사도 재계산**
- `train_filled = train_matrix.fillna(0)`
- 유저 유사도: `cosine_similarity(train_filled)` → `train_user_sim` (DataFrame)
- 아이템 유사도: `cosine_similarity(train_filled.T)` → `train_item_sim` (DataFrame)

**3단계: Train 기반 인기 상품 리스트**
- `train`에서 상품별 판매건수, 평균평점, 인기점수를 계산합니다
- 인기점수 순으로 정렬한 상품ID 리스트 → `popular_items`

사용할 함수/메서드: `sort_values()`, `tolist()`


In [ ]:
# Train 기반으로 유저-상품 매트릭스 재구성

# 아래에 코드를 작성하세요


#### 가이드: 평가용 추천 함수 구현

Train 기반 유저/아이템 추천 함수를 구현합니다. 상품ID 리스트만 반환하는 간소화 버전입니다.

**함수 1: `get_user_based_recs(user_id, train_user_sim, train_matrix, K=10, n_similar=10)`**
- 유사 유저 `n_similar`명의 가중 평점으로 추천
- 이미 구매한 상품 제외
- 예측 평점 상위 `K`개의 상품ID 리스트 반환

**함수 2: `get_item_based_recs(user_id, train_item_sim, train_matrix, K=10, n_similar=5)`**
- 해당 유저가 구매한 각 상품에 대해 유사 상품을 찾습니다
- 유사도와 기존 평점을 가중하여 점수를 계산합니다
- 이미 구매한 상품 제외 후 상위 `K`개 상품ID 리스트 반환

두 함수 모두 `user_id`가 train에 없으면 빈 리스트 `[]`를 반환합니다.



In [ ]:
# 아래에 코드를 작성하세요


#### 가이드: Hit Rate 및 Precision@K 성능 평가

**평가 지표 설명:**
- **Hit Rate**: 추천 리스트 중 1개라도 실제 구매한 상품이 있으면 Hit (비율)
- **Precision@K**: 추천 K개 중 실제 구매 상품의 비율

**1단계: 테스트셋 준비**
- `K = 10` (추천 개수)
- `test.groupby('유저ID')['상품ID'].apply(set).to_dict()` → `test_user_items`
- train에 존재하는 유저만 평가 대상으로 선별합니다

**2단계: 3가지 방식별 평가 반복**
각 유저에 대해:
1. **인기도 기반**: `popular_items`에서 이미 구매한 상품 제외 후 상위 K개
2. **유저 기반**: `get_user_based_recs()` 호출
3. **아이템 기반**: `get_item_based_recs()` 호출

**3단계: Hit/Precision 집계**
- `set(recs) & actual`로 교집합 → hit 수 계산
- 결과를 `perf_df` DataFrame으로 정리합니다



In [ ]:
# 성능 평가: Hit Rate, Precision@K

# 아래에 코드를 작성하세요


#### 가이드: 추천 성능 비교 시각화

**1x2 subplot 구성:**
- `fig, axes = plt.subplots(1, 2, figsize=(12, 5))`
- `axes[0]`: Hit Rate 비교 막대그래프
- `axes[1]`: Precision@K 비교 막대그래프
- 3가지 방식(`'인기도'`, `'유저기반'`, `'아이템기반'`)을 색상으로 구분합니다: `['steelblue', 'coral', 'seagreen']`
- 각 막대 위에 수치를 텍스트로 표시합니다

사용할 함수/메서드: `bar()`


In [ ]:
# 성능 비교 시각화

# 아래에 코드를 작성하세요


## 9. 비즈니스 활용


### 9-1. 추천 결과 종합


#### 가이드: 유저별 종합 추천 리스트 생성

샘플 유저 5명(`'U0001'`, `'U0010'`, `'U0050'`, `'U0100'`, `'U0200'`)에 대해 2가지 추천을 모두 수행합니다.

**각 유저마다:**

1. **유저 정보 출력**: 성별(`'성별'`), 연령(`'연령'`), 선호카테고리(`'선호카테고리'`)
2. **유저 기반 추천**: `recommend_user_based(uid, user_sim_df, user_item_matrix, products, n_recommend=5)`
   - 상품ID, 상품명, 카테고리, 예측평점 출력
3. **아이템 기반 추천**: 가장 최근 구매 상품을 기준으로 추천
   - `purchases[purchases['유저ID'] == uid].sort_values('구매일').iloc[-1]`로 마지막 구매 조회
   - `recommend_item_based(last_pid, item_sim_df, products, n_recommend=5)`
   - 상품ID, 상품명, 카테고리, 유사도 출력

사용할 함수/메서드: `sort_values()`


In [ ]:
# 유저별 종합 추천 리스트 생성 (샘플 5명)

# 아래에 코드를 작성하세요


### 9-2. 종합 대시보드


#### 가이드: 2x3 종합 대시보드 구성

`fig, axes = plt.subplots(2, 3, figsize=(18, 12))`로 6칸 대시보드를 만듭니다.

**각 칸 구성:**

| 위치 | 차트 종류 | 데이터 | 주요 코드 |
|------|-----------|--------|-----------|
| `axes[0,0]` | 파이차트 | 카테고리별 판매 비율 | `df['카테고리'].value_counts()` → `ax.pie()` |
| `axes[0,1]` | 수평 막대 | 인기 상품 Top 10 | `popularity.head(10)` → `ax.barh()` |
| `axes[0,2]` | 막대그래프 | 평점 분포 | `rating_counts` 변수 재사용 |
| `axes[1,0]` | 히스토그램 | 유저별 구매수 분포 | `user_stats['구매수']` 변수 재사용 |
| `axes[1,1]` | 그룹 막대 | 추천 성능 비교 (Hit Rate + Precision@K) | `perf_df` 변수 재사용 |
| `axes[1,2]` | 라인차트 | 월별 구매 추이 | `df['구매일'].dt.to_period('M')` |

**주요 패턴:**
사용할 함수/메서드: `bar()`, `groupby()`, `pie()`, `plot()`, `size()`, `to_period()`


In [ ]:
# 아래에 코드를 작성하세요


### 9-3. 서비스 적용 제안

| 적용 위치 | 추천 방식 | 설명 |
|-----------|-----------|------|
| **홈 화면** | 유저 기반 협업 필터링 | 로그인 유저에게 취향이 비슷한 유저들이 좋아하는 상품 추천 |
| **상품 상세 페이지** | 아이템 기반 협업 필터링 | "이 상품을 보신 분은 이런 상품도 좋아합니다" |
| **장바구니** | 아이템 기반 협업 필터링 | 장바구니에 담긴 상품과 유사한 상품 추가 추천 |
| **비로그인/신규 유저** | 인기도 기반 | 콜드스타트 문제 해결을 위해 전체/카테고리별 인기 상품 추천 |
| **마이페이지** | 유저 기반 + 아이템 기반 결합 | 개인화 추천 리스트 제공 |

**추가 개선 방향:**
- **하이브리드 추천:** 유저 기반 + 아이템 기반 + 컨텐츠 기반을 결합한 앙상블 모델
- **딥러닝 적용:** Neural Collaborative Filtering (NCF), AutoEncoder 기반 추천
- **실시간 추천:** 유저의 실시간 브라우징 행동을 반영한 세션 기반 추천
- **A/B 테스트:** 추천 방식별 CTR, 전환율 비교를 통한 최적 모델 선정
- **콜드스타트 개선:** 신규 유저 온보딩 시 선호 카테고리 설문으로 초기 추천 품질 향상
